In [ ]:
from itertools import product
import psutil
from functools import partial
import sys
# !{sys.executable} --version
# !{sys.executable} -m pip install shap --upgrade 
from qiskit_addon_sqd.counts import counts_to_arrays, bit_array_to_arrays,BitArray
from tqdm.notebook import tqdm
import joblib
import time
from shutil import copy
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, r2_score
import os
import matplotlib.pyplot as plt
from glob import glob
import seaborn as sns 
from tqdm.notebook import tqdm
from matplotlib import font_manager
import scipy.stats as stats
from shutil import move
font_path = 'Futura Book.ttf'
font_manager.fontManager.addfont(font_path)
prop = font_manager.FontProperties(fname=font_path, size='large')
plt.rcParams['font.family'] = prop.get_name()
plt.rcParams.update({'font.size': 12})

from qiskit_addon_sqd.subsampling import postselect_by_hamming_right_and_left


In [ ]:
datadf = pd.read_csv("../../../DDLUCJ_active_spaces_unfrozen.csv",delimiter=';').dropna(axis=1)

In [ ]:
UofT_palette = [ "#1E3765",
                 "#007FA3", 
                 "#6D247A", 
                 "#DC4633",
                 "#6FC7EA",
                 "#00A189",
                 "#AB1368",
                 "#0D534D",
                 "#F1C500",
                 "#8DBF2E"
               ]

palette = sns.color_palette(UofT_palette)

In [ ]:
import math


def fci_dimension(n_alpha: int, n_beta: int, n_orbitals: int) -> int:
    """Calculates the FCI space dimension for a given number of alpha/beta

    electrons and spatial orbitals.
    """
    alpha_combs = math.comb(n_orbitals, n_alpha)
    beta_combs = math.comb(n_orbitals, n_beta)

    return alpha_combs * beta_combs

In [ ]:
n_atoms_dict = {
    'water': 3,
    'methane': 5,
    'ammonia': 4,
    'ethane': 8,
    'methanol': 6,
    'ethylene': 6,
    'formaldehyde': 4,
    "prop-2-en-1-ol":10,
    "but-1-yne":10,
    "fluoroform":5,
    "buta-1,3-diene":10,
    "(Z)-1-fluoroprop-1-ene":9
}



In [ ]:
dim_data = []
for i in sorted(glob("./counts/*npz")):
    parts = os.path.basename(i).replace('.npz', '').split('_')
    name, _, rawlayers, basis = parts[:4]
    injection = '_'.join(parts[4:])

    n_elec = datadf.loc[datadf['molecule'] == name, 'Ne'].values[0]
    n_orb = datadf.loc[datadf['molecule'] == name, 'No'].values[0]
    right = left = n_elec // 2
    layers = int(rawlayers.strip("L"))
    exact_dim = fci_dimension(left,right,n_orb)
    counts = np.load(i)
    probarr = counts['probarr']
    bitstrings = counts['bitstrings']  # already the right shape/dtype, no conversion needed

    ps_bitstrings, ps_probs = postselect_by_hamming_right_and_left(
        bitstrings, probarr,
        hamming_right=right, hamming_left=left,
    )
    
    dim_data.append((name, layers, basis, injection, ps_bitstrings.shape[0],bitstrings.shape[0],exact_dim,10000,n_orb,n_elec))

dim_df = pd.DataFrame(dim_data,columns=['Name','L','Basis','Injection','Postselected','Device','Exact','Shots','NOrb','Nelec'])

# 1. Format ActiveSpace strings
dim_df['ActiveSpace'] = dim_df.apply(lambda r: f"({int(r['Nelec'])}, {int(r['NOrb'])})", axis=1)
dim_df['n_atoms'] = dim_df['Name'].map(n_atoms_dict)

dim_df['PostselectedPercent'] = (dim_df['Postselected'] / dim_df['Exact'])*1e2

dim_df.to_excel("Dimensions.xlsx")

In [ ]:
np.unique(dim_df['Exact'])

In [ ]:
f"{59693548345600:.2e}"

In [ ]:
SIZE = 14

plt.rc('font', size=SIZE)          # controls default text sizes
plt.rc('axes', titlesize=SIZE)     # fontsize of the axes title
plt.rc('axes', labelsize=SIZE)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=SIZE)    # fontsize of the tick labels
plt.rc('ytick', labelsize=SIZE)    # fontsize of the tick labels
plt.rc('legend', fontsize=SIZE)    # legend fontsize
plt.rc('figure', titlesize=SIZE)  # fontsize of the figure title

# 2. Sort dataframe
df_sorted = dim_df.sort_values(by=["NOrb", "Nelec", "Basis", "L"])

# Get unique Injection categories to slice palette and set hue order
injection_order = ['zeroes', 'random', 'MP2', 'ML', 'ML_exact', 'CCSD']
palette_sliced = palette[:len(injection_order)] if isinstance(palette, list) else palette

basis_order = ['STO-3G', 'cc-pVDZ', 'aug-cc-pVDZ']

# 1. Changed to 3 rows and adjusted height to 11
fig, axes = plt.subplots(3, 3, figsize=(16, 11), sharex=True, sharey='row')

# Loop through each Basis column and plot Top (Device), Middle (Postselected), and Bottom (PostselectedPercent)
for i, basis in enumerate(basis_order):
    sub_df = df_sorted[df_sorted['Basis'] == basis]
    
    # Top Row: Device
    sns.lineplot(
        data=sub_df, x='ActiveSpace', y='Device', hue='Injection', style='Injection',
        hue_order=injection_order, style_order=injection_order,
        markers=True, palette=palette_sliced, ax=axes[0, i], legend=False
    )
    axes[0, i].set_title(f"Basis = {basis}")
    
    # Middle Row: Postselected
    sns.lineplot(
        data=sub_df, x='ActiveSpace', y='Postselected', hue='Injection', style='Injection',
        hue_order=injection_order, style_order=injection_order,
        markers=True, palette=palette_sliced, ax=axes[1, i], legend=False
    )

    # 2. Bottom Row: PostselectedPercent
    sns.lineplot(
        data=sub_df, x='ActiveSpace', y='PostselectedPercent', hue='Injection', style='Injection',
        hue_order=injection_order, style_order=injection_order,
        markers=True, palette=palette_sliced, ax=axes[2, i], legend=(i == 0)
    )
    
    # Rotate x-axis labels on the NEW bottom row (index 2) for readability
    axes[2, i].tick_params(axis='x', rotation=45)

# Extract legend elements from the plot that generated it (row 2, col 0)
handles, labels = axes[2, 0].get_legend_handles_labels()
axes[2, 0].get_legend().remove()

# Add a single shared figure legend
fig.legend(
    handles, labels, 
    title='Injection', 
    loc='center left', 
    bbox_to_anchor=(0.89, 0.5), 
    frameon=False
)

# Label x-axis cleanly ONLY on bottom subplots (row 2)
for ax in axes[2, :]:
    ax.set_xlabel("Active Spaces ($N_{e}$, $N_{o}$)")

# Set Y limits
axes[0,0].set_ylim(0, 10200)
axes[1,0].set_ylim(0, 2100)
# Optional: Set y-limits for percent row if values are 0–100 or 0–1 (e.g., 0–100%)
axes[2,0].set_ylim(0, 100)

# Set Y labels
axes[0,0].set_ylabel(r"Unique Configurations ($ibm\_quebec$)")
axes[1,0].set_ylabel(r"Post-Selected Configurations")
axes[2,0].set_ylabel("FCI Coverage (%)")  # Added 3rd row label

plt.tight_layout()
fig.subplots_adjust(right=0.88) # Leaves room on the right for legend
plt.savefig("./GMJ_figures/ConfigsVsActiveSpaces.png", dpi=300, bbox_inches='tight')
plt.show()